# EDS Test Data Export — Original Inputs & Derived Outputs

## Purpose

This notebook prepares **small, reproducible ZIP bundles** of Environmental Data Science (EDS) inputs and outputs for **download, inspection, and validation**.

It is designed for **testing, auditing, and troubleshooting** EDS processing chains (e.g. *SR → NDVI / FC compatibility products*) without requiring access to the full EASI scratch environment.

The export focuses on:

- **Original source rasters** (Surface Reflectance or Fractional Cover)
- **Derived compatibility outputs**
- **All required ENVI sidecar files** (`.hdr`, `.aux.xml`, `.ovr`, `.prj`)

---

## What This Notebook Does

For a selected EDS run (defined by a JSON run log), this notebook:

### 1. Reads the EDS JSON manifest
- Identifies the processing tile (e.g. `p089r078`)
- Locates original input datasets (SR or FC)
- Locates derived outputs under the compatibility directory

### 2. Randomly selects a small subset of outputs
- Defaults to **5 output rasters** per run
- Samples only **primary raster files** (`.img`)
- Automatically includes all associated ENVI sidecars

### 3. Matches outputs to original inputs
- **FC workflows**: matches against `inputs.fc.matched_files`
- **SR / NDVI workflows**: reconstructs expected SR paths from `{scene + date}`

### 4. Stages and exports a ZIP archive
- Preserves **tile-relative directory structure** inside the ZIP
- Writes ZIP files to:

---

## Why This Exists

This export workflow supports:

- Validation of processing correctness
- Spot-checking of time-series behaviour
- Inspection of NoData handling and masking
- Lightweight sharing of test cases
- Offline review without EASI access

It intentionally avoids exporting:

- Full time series
- Large DC4 stacks (unless explicitly required)
- Redundant intermediate products


In [56]:
timeseries_data = "fc"

if timeseries_data == "ndvi":
    data = "sr"
else:
    data = "fc"

In [57]:
from pathlib import Path
import json
import re
import random
import shutil
import zipfile
from datetime import datetime

def load_json(path: str | Path) -> dict:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def safe_copy(src: str | Path, dst_dir: str | Path) -> Path | None:
    src = Path(src)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        print(f"[MISS] {src}")
        return None
    dst = dst_dir / src.name
    if dst.exists():
        # keep existing (no overwrite)
        return dst
    shutil.copy2(src, dst)
    return dst

def zip_paths(zip_path: str | Path, paths: list[Path]) -> Path:
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if p is None:
                continue
            p = Path(p)
            if not p.exists():
                continue
            # store only filename in the zip (change to p.as_posix() if you want full paths)
            z.write(p, arcname=p.name)
    return zip_path

def extract_scene_and_date_from_name(name: str) -> tuple[str | None, str | None]:
    """
    Tries to pull:
      - scene like p089r078
      - date like 20230829
    from filenames like:
      lztmre_p089r078_20230829_db8mz.img
      lztmre_p089r078_20160318_dc4fc.img
    """
    scene = None
    date = None
    m_scene = re.search(r"(p\d{3}r\d{3})", name.lower())
    if m_scene:
        scene = m_scene.group(1)
    m_date = re.search(r"(19|20)\d{6}", name)
    if m_date:
        date = m_date.group(0)
    return scene, date

def pick_random_files(root: str | Path, n: int = 5, pattern: str = "*") -> list[Path]:
    root = Path(root)
    files = [p for p in root.rglob(pattern) if p.is_file()]
    if not files:
        return []
    n = min(n, len(files))
    return random.sample(files, n)

def match_fc_input_by_date(fc_matched_files: list[str], target_date: str) -> str | None:
    """
    Best-effort match: find an FC original path that contains the same YYYYMMDD.
    """
    if not target_date:
        return None
    for p in fc_matched_files:
        if target_date in p:
            return p
    return None


In [58]:
# from glob import glob

# ROOT = f"/home/jovyan/work-easi-eds/data/compat/files/{timeseries_data}"
# cands = sorted(glob(f"{ROOT}/**/*.json", recursive=True))
# print("Found JSON files:", len(cands))
# print("\n".join(cands[:30]))


In [59]:
# # Set this to your JSON file
# JSON_PATH = "/home/jovyan/work-easi-eds/data/compat/files/fc/p089r078/eds_master_results_089_078_fc_d20230720_20240831.json"

# data = load_json(JSON_PATH)
# print("[OK] Loaded JSON keys:", list(data.keys()))


In [60]:
from glob import glob
from pathlib import Path

ROOT = f"/home/jovyan/work-easi-eds/data/compat/files/{timeseries_data}"

json_paths = sorted(glob(f"{ROOT}/**/*.json", recursive=True))

print(f"Found JSON files: {len(json_paths)}\n")

for i, p in enumerate(json_paths):
    print(f"[{i}] {p}")


Found JSON files: 8

[0] /home/jovyan/work-easi-eds/data/compat/files/fc/p089r078/eds_master_results_089_078_fc_d20230306_20231024.json
[1] /home/jovyan/work-easi-eds/data/compat/files/fc/p089r078/lztmre_p089r078_d2023030620231024_vi-fc_dllmz_log.json
[2] /home/jovyan/work-easi-eds/data/compat/files/fc/p089r084/eds_master_results_089_084_fc_d20230125_20231024.json
[3] /home/jovyan/work-easi-eds/data/compat/files/fc/p089r084/lztmre_p089r084_d2023012520231024_vi-fc_dllmz_log.json
[4] /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/eds_master_results_100_082_fc_d20230421_20231022.json
[5] /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_d2023042120231022_vi-fc_dllmz_log.json
[6] /home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/eds_master_results_104_074_fc_d20230503_20231026.json
[7] /home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/lztmre_p104r074_d2023050320231026_vi-fc_dllmz_log.json


In [67]:
# 👇 Change this number to select a different JSON
SELECT_INDEX = 6

JSON_PATH = Path(json_paths[SELECT_INDEX])
data = load_json(JSON_PATH)

print("[OK] Selected JSON:")
print(JSON_PATH)
print("\nTop-level keys:")
print(list(data.keys()))


[OK] Selected JSON:
/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/eds_master_results_104_074_fc_d20230503_20231026.json

Top-level keys:
['cwd', 'effective_end_date', 'effective_start_date', 'fc_inputs', 'hostname', 'inputs', 'outputs', 'outputs_root', 'requested_end_date', 'requested_start_date', 'scene', 'seasonal_window', 'sr_inputs', 'steps', 'tile', 'timeseries_source', 'timestamp_utc']


In [68]:
# Try to normalise to a list of "records"
records = []
if isinstance(data, list):
    records = data
elif isinstance(data, dict) and "records" in data and isinstance(data["records"], list):
    records = data["records"]
else:
    records = [data]

print("[INFO] Records:", len(records))


[INFO] Records: 1


In [69]:
from pathlib import Path
import shutil

def sidecar_paths(p: str | Path) -> list[Path]:
    """
    Return a list including p + common sidecars (only if they exist).
    Works for:
      - ENVI .img  → .hdr (+ optional .aux.xml/.ovr)
      - GeoTIFF .tif/.tiff → .aux.xml/.ovr
      - General → .aux.xml/.ovr (if present)
    """
    p = Path(p)
    candidates = [p]

    # Common GDAL sidecars for many rasters
    candidates.append(p.with_name(p.name + ".aux.xml"))  # e.g. file.tif.aux.xml or file.img.aux.xml
    candidates.append(p.with_name(p.name + ".ovr"))      # e.g. file.tif.ovr or file.img.ovr

    # ENVI header for .img
    if p.suffix.lower() == ".img":
        candidates.append(p.with_suffix(".hdr"))
        candidates.append(p.with_suffix(".HDR"))

    # Some workflows create .prj next to ENVI rasters
    candidates.append(p.with_suffix(".prj"))
    candidates.append(p.with_suffix(".PRJ"))

    # Return only those that exist, de-duped
    out = []
    seen = set()
    for c in candidates:
        if c.exists():
            cp = c.resolve()
            if cp not in seen:
                out.append(c)
                seen.add(cp)
    return out

def safe_copy_with_sidecars(src: str | Path, dst_dir: str | Path) -> list[Path]:
    """
    Copies src and any detected sidecars into dst_dir (no overwrite).
    Returns list of copied/existing destination Paths.
    """
    src = Path(src)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print(f"[MISS] {src}")
        return []

    copied = []
    for f in sidecar_paths(src):
        dst = dst_dir / f.name
        if dst.exists():
            copied.append(dst)
            continue
        shutil.copy2(f, dst)
        copied.append(dst)
    return copied


In [70]:
# Where to stage copies before zipping
STAGE_DIR = Path("/home/jovyan/work-easi-eds/exports/stage_sr")
ZIP_DIR   = Path("/home/jovyan/work-easi-eds/exports/zips")
ZIP_DIR.mkdir(parents=True, exist_ok=True)

all_sr_copied = []

for i, rec in enumerate(records, start=1):
    sr = (rec.get("inputs") or {}).get("sr") or {}
    sr_start = (sr.get("start") or {}).get("path")
    sr_end   = (sr.get("end") or {}).get("path")

    print(f"\n[REC {i}] SR start:", sr_start)
    print(f"[REC {i}] SR end  :", sr_end)

    copied = []
    if sr_start:
        copied.extend(safe_copy_with_sidecars(sr_start, STAGE_DIR))
    if sr_end:
        copied.extend(safe_copy_with_sidecars(sr_end, STAGE_DIR))


    copied = [p for p in copied if p is not None]
    all_sr_copied.extend(copied)

# Zip them
sr_zip = ZIP_DIR / f"sr_start_end_SR_data_from_{Path(JSON_PATH).stem}.zip"
zip_paths(sr_zip, all_sr_copied)

print("\n[OK] SR files copied:", len(all_sr_copied))
print("[OK] SR zip:", sr_zip)



[REC 1] SR start: /home/jovyan/scratch/eds/tiles/p104r074/sr/2023/202305/ls89sr_p104r074_20230503_nbart6m2_clr.tif
[REC 1] SR end  : /home/jovyan/scratch/eds/tiles/p104r074/sr/2023/202310/ls89sr_p104r074_20231026_nbart6m2_clr.tif

[OK] SR files copied: 2
[OK] SR zip: /home/jovyan/work-easi-eds/exports/zips/sr_start_end_from_json_eds_master_results_104_074_fc_d20230503_20231026.zip


In [71]:
# STAGE_DIR_OUT = Path("/home/jovyan/work-easi-eds/exports/stage_outputs_and_inputs")
# STAGE_DIR_OUT.mkdir(parents=True, exist_ok=True)

# # pick outputs_root from the first record if present; fallback to your known path
# outputs_root = records[0].get("outputs_root") or f"/home/jovyan/work-easi-eds/data/compat/files/{data}"

# outputs_root = Path(outputs_root)
# print(outputs_root)
# # gather FC matched_files (first record)
# matched_files = ((records[0].get("inputs") or {}).get(data) or {}).get("matched_files") or []
# print("[INFO] outputs_root:", outputs_root)
# print(f"[INFO] {data} matched_files:", len(matched_files))

# # 1) random 5 outputs (recursively)
# random_outputs = pick_random_files(outputs_root, n=5, pattern="*")
# print("\nRandom outputs:")
# for p in random_outputs:
#     print(" -", p)

# copied_everything = []

# # 2) copy outputs + try match & copy original FC inputs
# for outp in random_outputs:
#     # copy the output
#     copied_out = safe_copy(outp, STAGE_DIR_OUT)
#     if copied_out:
#         copied_everything.append(copied_out)

#     # infer date from output filename
#     scene, date = extract_scene_and_date_from_name(outp.name)
#     fc_src = match_fc_input_by_date(fc_matched_files, date) if date else None

#     print(f"\n[OUT] {outp.name}")
#     print("  scene:", scene, "date:", date)
#     print(f"  matched {data} input:", fc_src)

#     if fc_src:
#         copied_fc = safe_copy(fc_src, STAGE_DIR_OUT)
#         if copied_fc:
#             copied_everything.append(copied_fc)

# # 3) zip the staged selection
# bundle_zip = ZIP_DIR / f"random5_db4_outputs_plus_{data}_inputs_{Path(JSON_PATH).stem}.zip"
# zip_paths(bundle_zip, copied_everything)

# print("\n[OK] Copied total:", len(copied_everything))
# print("[OK] Bundle zip:", bundle_zip)


In [72]:
from pathlib import Path
import re

STAGE_DIR_OUT = Path("/home/jovyan/work-easi-eds/exports/stage_outputs_and_inputs")
STAGE_DIR_OUT.mkdir(parents=True, exist_ok=True)

ZIP_DIR = Path("/home/jovyan/work-easi-eds/exports/zips")
ZIP_DIR.mkdir(parents=True, exist_ok=True)

def extract_scene_and_date_from_name(name: str) -> tuple[str | None, str | None]:
    scene = None
    date = None
    m_scene = re.search(r"(p\d{3}r\d{3})", name.lower())
    if m_scene:
        scene = m_scene.group(1)
    m_date = re.search(r"(19|20)\d{6}", name)
    if m_date:
        date = m_date.group(0)
    return scene, date

def guess_sr_path(scene: str, date: str) -> Path:
    """
    Matches your EDS SR layout:
      /home/jovyan/scratch/eds/tiles/{scene}/sr/{YYYY}/{YYYYMM}/ls89sr_{scene}_{YYYYMMDD}_nbart6m6_clr.tif
    """
    yyyy = date[:4]
    yyyymm = date[:6]
    return Path(f"/home/jovyan/scratch/eds/tiles/{scene}/sr/{yyyy}/{yyyymm}/ls89sr_{scene}_{date}_nbart6m6_clr.tif")

def match_input_for_output(data: str, matched_files: list[str], scene: str | None, date: str | None) -> str | None:
    """
    - FC: match within matched_files by YYYYMMDD
    - SR: derive expected SR path and check it exists
    """
    if not date or not scene:
        return None

    if data == "fc":
        for p in matched_files:
            if date in p:
                return p
        return None

    if data == "sr":
        p = guess_sr_path(scene, date)
        return str(p) if p.exists() else None

    # fallback: try same behaviour as fc (date contains)
    for p in matched_files:
        if date in p:
            return p
    return None

# outputs_root from JSON (or fallback)
outputs_root = records[0].get("outputs_root") or f"/home/jovyan/work-easi-eds/data/compat/files/{data}"
outputs_root = Path(outputs_root)

# # matched_files list for FC only (SR usually doesn't have it)
# matched_files = ((records[0].get("inputs") or {}).get(data) or {}).get("matched_files") or []

# print("[INFO] outputs_root:", outputs_root)
# print(f"[INFO] {data} matched_files:", len(matched_files))

# manifest = loaded JSON dict
manifest = records[0]  # or manifest = load_json(JSON_PATH) if you prefer

# timeseries_data should be a string like "fc" or "sr"
# (If you already have timeseries_data, keep it. Otherwise set it here.)
# timeseries_data = "fc"  # example

outputs_root = manifest.get("outputs_root") or f"/home/jovyan/work-easi-eds/data/compat/files/{timeseries_data}"
outputs_root = Path(outputs_root)

matched_files = ((manifest.get("inputs") or {}).get(timeseries_data) or {}).get("matched_files") or []

print("[INFO] outputs_root:", outputs_root)
print(f"[INFO] {timeseries_data} matched_files:", len(matched_files))

random_outputs = [p for p in pick_random_files(outputs_root, n=20, pattern="*.img")]  # grab up to 20 candidates
random_outputs = random_outputs[:5]  # take first 5 from the random sample list

print("\nRandom outputs (.img only):")
for p in random_outputs:
    print(" -", p)

copied_everything = []

# for outp in random_outputs:
#     # copy output + sidecars
#     copied_everything.extend(safe_copy_with_sidecars(outp, STAGE_DIR_OUT))

#     scene, date = extract_scene_and_date_from_name(outp.name)
#     src_input = match_input_for_output(timeseries_data, matched_files, scene, date)
#     print(f"  matched {timeseries_data} input:", src_input)

#     print("  scene:", scene, "date:", date)
#     print(f"  matched {data} input:", src_input)

#     if src_input:
#         copied_everything.extend(safe_copy_with_sidecars(src_input, STAGE_DIR_OUT))


for outp in random_outputs:
    # copy output + sidecars
    copied_everything.extend(safe_copy_with_sidecars(outp, STAGE_DIR_OUT))

    scene, date = extract_scene_and_date_from_name(outp.name)
    src_input = match_input_for_output(timeseries_data, matched_files, scene, date)

    print(f"\n[OUT] {outp.name}")
    print("  scene:", scene, "date:", date)
    print(f"  matched {timeseries_data} input:", src_input)

    if src_input:
        copied_everything.extend(safe_copy_with_sidecars(src_input, STAGE_DIR_OUT))

# bundle_zip = ZIP_DIR / f"random5_{timeseries_data}_outputs_plus_{data}_inputs_{Path(JSON_PATH).stem}.zip"
bundle_zip = ZIP_DIR / f"random5_{timeseries_data}_outputs_plus_{timeseries_data}_inputs_{Path(JSON_PATH).stem}.zip"



zip_paths(bundle_zip, copied_everything)

print("\n[OK] Copied total:", len(copied_everything))
print("[OK] Bundle zip:", bundle_zip)


[INFO] outputs_root: /home/jovyan/work-easi-eds/data/compat/files/fc
[INFO] fc matched_files: 243

Random outputs (.img only):
 - /home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/lztmre_p104r074_20180606_dc4fc.img
 - /home/jovyan/work-easi-eds/data/compat/files/fc/p089r078/lztmre_p089r078_20211026_dc4fc.img
 - /home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/lztmre_p104r074_20220703_dc4fc.img
 - /home/jovyan/work-easi-eds/data/compat/files/fc/p089r084/lztmre_p089r084_20181205_dc4fc.img
 - /home/jovyan/work-easi-eds/data/compat/files/fc/p089r084/lztmre_p089r084_d2023012520231024_vi-fc_dllmz.img

[OUT] lztmre_p104r074_20180606_dc4fc.img
  scene: p104r074 date: 20180606
  matched fc input: /home/jovyan/scratch/eds/tiles/p104r074/fc/2018/201806/galsfc3_p104r074_20180606_fcm2_clr.tif

[OUT] lztmre_p089r078_20211026_dc4fc.img
  scene: p089r078 date: 20211026
  matched fc input: None

[OUT] lztmre_p104r074_20220703_dc4fc.img
  scene: p104r074 date: 20220703
  matched fc input: